# `consolidate.ipynb` - Consolidación estadística y cálculo de indicadores

Requisito: *"La base de datos local contiene miles de reportes duplicados por país con fluctuaciones. Aplica el método estadístico adecuado para cancelar el ruido numérico y obtener una cifra única representativa por país."*

Se agrupa por país (ya normalizado) y se calcula la **mediana** de `superficie_km2` y `pib_total_eur`. Se elige la mediana y no la media porque es un estadístico robusto: no se deja arrastrar por los valores atípicos que pudieran quedar tras el filtro de cuarentena.

Requisito: *"Genera para cada país la Densidad Poblacional (habitantes/km²) y el PIB per Cápita (€/habitante)."*

> Solo depende de `pandas` - no necesita `config.ipynb` ni PostgreSQL, así que las pruebas de este notebook usan un DataFrame de ejemplo pequeño en vez de los datos reales.

In [ ]:
import logging

import pandas as pd

## 1. Consolidación estadística (mediana)

In [ ]:
def consolidar_estadisticamente(df_validos: pd.DataFrame) -> pd.DataFrame:
    """
    La fuente Legacy contiene miles de reportes duplicados por pais con
    fluctuaciones de ruido. Se aplica la MEDIANA (estadistico robusto,
    poco sensible a valores atipicos frente a la media) para obtener
    una cifra unica representativa de PIB y superficie por pais.
    """
    logger.info("Consolidando registros duplicados por pais mediante la mediana...")
    df_consolidado = df_validos.groupby("nombre_pais", as_index=False).agg(
        superficie_km2=("superficie_km2", "median"),
        pib_total_eur=("pib_total_eur", "median"),
        n_reportes=("pib_total_eur", "count"),
    )
    logger.info(f"Consolidacion completada: {len(df_consolidado)} paises unicos.")
    return df_consolidado

## 2. Cálculo de indicadores

In [ ]:
def calcular_indicadores(df_consolidado: pd.DataFrame, poblacion: dict, origen_poblacion: str) -> pd.DataFrame:
    """Cruza con los datos demograficos y calcula densidad y PIB per capita."""
    df = df_consolidado.copy()
    df["poblacion_total"] = df["nombre_pais"].map(poblacion)

    sin_poblacion = df[df["poblacion_total"].isna()]["nombre_pais"].tolist()
    if sin_poblacion:
        logger.warning(f"Paises sin dato de poblacion disponible (se excluyen del destino): {sin_poblacion}")
    df = df.dropna(subset=["poblacion_total"]).copy()

    df["poblacion_total"] = df["poblacion_total"].astype("int64")
    df["densidad_poblacional"] = (df["poblacion_total"] / df["superficie_km2"]).round(4)
    df["pib_per_capita_eur"] = (df["pib_total_eur"] / df["poblacion_total"]).round(2)
    df["fuente_poblacion"] = origen_poblacion

    # Nombre "bonito" (Title Case) para guardar en destino, en vez de la clave normalizada en mayúsculas
    df["nombre_pais"] = df["nombre_pais"].str.title()
    return df

## Prueba rápida

Un DataFrame de ejemplo con duplicados ruidosos de dos países (simula lo que llega desde `extract.ipynb`), para comprobar que la mediana cancela el ruido y que los indicadores salen coherentes, sin depender de la SQLite real ni de PostgreSQL.

In [ ]:
df_ejemplo = pd.DataFrame({
    "nombre_pais": ["MONACO", "MONACO", "MONACO", "FRANCE", "FRANCE", "FRANCE"],
    "superficie_km2": [2.0, 2.0, 2.0, 643801.0, 643801.0, 643801.0],
    "pib_total_eur": [8.50e9, 8.52e9, 8.55e9, 2.78e12, 2.79e12, 2.80e12],  # ruido tipico de duplicados
})

df_consolidado_prueba = consolidar_estadisticamente(df_ejemplo)
print(df_consolidado_prueba.to_string(index=False))

poblacion_ejemplo = {"MONACO": 38_341, "FRANCE": 66_650_804}
df_indicadores_prueba = calcular_indicadores(df_consolidado_prueba, poblacion_ejemplo, "FALLBACK")
print()
print(df_indicadores_prueba.to_string(index=False))